# Assignment 5 — The Best Portfolio, and How Little You Know About It


Graded on completion. Groups of three; each of you submits your own copy.

---

**Group members**

- Name / NYU email:
- Name / NYU email:
- Name / NYU email:

---

This one is **not** about your group's strategy. It uses the 49 industries from
Assignment 1, because the problem it is about only shows up with many assets.

*Capital Allocation I* gave you one formula for the portfolio with the highest
Sharpe ratio: weights proportional to $\Sigma^{-1}\mu$. You know neither input.
You estimate both from the same data you then judge the portfolio on. This
assignment measures what that costs.

- **Part 1** — how well do you know the 49 means?
- **Part 2** — the tangency portfolio, and what moving one mean does to it
- **Part 3** — two simulated worlds where you know the answer: what would someone
  with your data find?
- **Part 4** — the real test: estimate on the first half, hold it through the second

## 🛠️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
ind  = pd.read_parquet(f"{BASE}/ind49_monthly_1926.parquet")      # 49 value-weighted industries, monthly
fac  = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")    # Mkt-RF and RF, from Ken French

df  = ind.sub(fac['RF'], axis=0).dropna()     # excess returns; only months in which all 49 exist
mkt = fac['Mkt-RF'].loc[df.index]             # the market, on the same months

T, N = df.shape
sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)      # annualized, from monthly returns
rng = np.random.default_rng(54)                          # use this for every random draw

print(f"{N} industries, {T} months: {df.index[0]:%Y-%m} to {df.index[-1]:%Y-%m}")

> **📌 Units.** Returns are **decimals per month**: 0.01 is 1%. The Assignment 1
> file was in percent; this one is not.
>
> **📌 Why 1969.** `dropna()` keeps only the months in which all 49 industries
> have a return. A few industries — Healthcare is the last — only start in 1969.

---

# Part 1 — How well do you know the means?

### Q1 — Means, standard errors, and whether you can tell them apart

For each industry compute the sample mean excess return, its standard error

$$\operatorname{se}(\bar r_i) = \frac{\sigma_i}{\sqrt{T}}$$

and the 95% confidence interval $\bar r_i \pm 1.96\, \operatorname{se}(\bar r_i)$.
The formula assumes returns are uncorrelated from one month to the next — not a
bad assumption for monthly returns. Annualize all of it (times 12) and keep it in
one DataFrame, `ci`, one row per industry, with columns `mean`, `se`, `lower`,
`upper`.

Report how many of the 49 intervals exclude zero, the spread of the 49 means
(their standard deviation across industries), and the median standard error.

**Can you tell them apart?** The tangency portfolio does not care whether
industries earn a premium. It cares which ones earn *more than the others*. So
build each industry's return minus the average of all 49 in the same month, and
test whether the mean of that difference is zero, with the same standard-error
formula.

> **🤔 Predict first.** Suppose no industry truly earned more than the average.
> At the 95% level, how many of 49 would "differ" anyway, by luck? Write it down.

In [ ]:
# your code here

**Predicted by luck:** · **intervals that exclude zero:** · **spread of the means:** · **median standard error:**

**Industries that differ from the average:**

**You know most industries earn a premium. Do you know which earn more?**

---

# Part 2 — The tangency portfolio

### Q2 — The weights

From *Capital Allocation I*: the portfolio with the highest Sharpe ratio has
weights proportional to

$$w \;\propto\; \Sigma^{-1} \mu$$

Use the sample means `mu = df.mean()` and the sample covariance `S = df.cov()`,
both monthly. Then scale the weights so the portfolio runs **10% annualized
volatility**: the monthly variance of a portfolio $w$ is $w'\Sigma w$, so multiply
$w$ by $0.10 / \sqrt{12\, w'\Sigma w}$.

You will build this portfolio about 2,000 times before you are done, so it is
worth writing once as a function that takes a mean vector and a covariance
matrix and returns the scaled weights.

Store the weights in a DataFrame `W`, one row per industry, in a column called
`'sample'`. Report the portfolio's in-sample Sharpe ratio, the Sharpe ratio of
**1/N** (equal weight in all 49) and of the market, the number of short
positions, and the gross exposure $\sum_i |w_i|$.

> **✅ Check.** The Sharpe ratio does not depend on the scaling, and it equals
> $\sqrt{12\, \mu'\Sigma^{-1}\mu}$. If yours does not match to three decimals,
> find out why before going on.

In [ ]:
# your code here

**Sharpe — tangency:** · **1/N:** · **market:** · **short positions:** · **gross exposure:**

**Given Q1, what is the optimizer using to get its Sharpe ratio?**

### Q3 — Move one mean

Take Household Products, `Hshld`. Replace its mean — only its — with the **lower
end** of its 95% interval, and recompute all 49 weights: same covariance, same
10% scaling. Then do it again with the **upper end**. Add the two as columns
`'Hshld low'` and `'Hshld high'` in `W`, and bar-plot the three columns.

> **📌 Units.** `ci` is annual; `mu` is monthly.

> **🤔 Predict first.** Which *other* industries' weights will move the most?
> Write down your guess before you look.

In [ ]:
# your code here

**Predicted:** · **Hshld's weight — sample:** · **low:** · **high:**

**The four other industries that move most, and why those four:**

### Q4 — What it costs

Compute the Sharpe ratio of each column of `W` **using the sample moments** `mu`
and `S` — as if the sample were the truth. Under that truth `'sample'` is the
best you can do, and the other two were built with a wrong mean for Hshld:

$$SR(w) = \sqrt{12}\; \frac{w'\mu}{\sqrt{w'\Sigma w}}$$

Now do it for **every** industry: loop over the 49, move each one's mean to the
lower and to the upper end of its interval (one at a time, the other 48 at their
sample values), and record the average percentage loss in Sharpe ratio:

$$\text{loss}_i \;=\; 1 - \frac{\tfrac12\left[\,SR(i\ \text{low}) + SR(i\ \text{high})\,\right]}{SR(\text{sample})}$$

Store it as a Series `loss` and bar-plot it, sorted.

In [ ]:
# your code here

**Hshld's loss:** · **average across the 49:** · **worst industry and its loss:**

**You moved one mean of 49, by an amount your data cannot rule out. Is the loss
large or small? What happens when all 49 are off at once — which is what
estimating them actually does?**

---

# Part 3 — Two worlds where you know the answer

In real data you never learn the true means, so you can never check how far off
you are. A simulation gets around that: **declare** a truth, draw fake histories
from it, and see what someone with 686 months of data would have found.

To draw one fake history of the same length as yours:

```python
X = pd.DataFrame(rng.multivariate_normal(mean_vector, cov_matrix, size=T), columns=df.columns)
```

### Q5 — World A: your sample is the truth

Treat `mu` and `S` as the true moments. Then the best portfolio is known: it is
`W['sample']`, with a Sharpe ratio of 1.20.

Draw **1,000** fake histories. In each one record three numbers:

| | what it is | what it stands for |
|---|---|---|
| **fixed** | the Sharpe ratio, in the fake history, of the true weights `W['sample']` | holding the right portfolio |
| **in-sample** | re-estimate the means and covariance from the fake history, build the tangency from them, and take its Sharpe ratio *in that same history* | what your backtest shows you |
| **true** | the Sharpe ratio of those re-estimated weights under the truth, `sr(w_hat, mu, S)` | what that portfolio actually earns |

Keep them in a DataFrame `mc`, one row per history, and plot the three
histograms on one figure.

> **✅ Check.** The average of `fixed` should be close to the 1.20 from Q2.

In [ ]:
# your code here

**Mean and 5–95% range — fixed:** · **in-sample:** · **true:**

**(a)** Even holding the right portfolio, how precisely do 57 years pin down its
Sharpe ratio?

**(b)** How much does the backtest overstate what the estimated portfolio
delivers? Can `true` ever exceed 1.20? Why not?

### Q6 — World B: nothing to find

Now a different truth. Keep `S`, but give **every industry the same mean**: the
average of the 49 means. In this world no industry earns more than another; the
only thing the optimizer can exploit is the covariance.

> **🤔 Predict first.** In World B, what in-sample Sharpe ratio will the estimated
> tangency show, on average? Write a number down.

Rerun `in-sample` and `true` from Q5 — 1,000 histories — with this truth. Also
compute, under World B's moments, the Sharpe ratio of 1/N and the best achievable
one (the tangency built from the true, equal means).

In [ ]:
# your code here

**Predicted:** · **in-sample (mean):** · **true (mean):** · **1/N:** · **best achievable:** · **share of histories in which the estimated tangency beats 1/N:**

**Compare the average in-sample Sharpe ratio with the 1.20 you found in Q2. What
does that tell you about the 1.20?**

---

# Part 4 — The real test

### Q7 — Estimate on the first half, hold it through the second

Part 3 needed a declared truth. Real data has a cleaner test, the one from
*Backtesting*: estimate on one sample, judge on another.

Split the sample down the middle — **estimate** 1969-07 to 1997-12, **hold**
1998-01 to 2026-08. Using the first half only, build the tangency weights at 10%
volatility. Then hold those weights, unchanged, through the second half.

> **🤔 Predict first.** In the second half, which has the higher Sharpe ratio —
> the tangency or 1/N?

Report the tangency's Sharpe ratio in the first half, and in the second; the
Sharpe ratio of 1/N and of the market in the second half; and the volatility the
tangency actually ran in the second half.

In [ ]:
# your code here

**Predicted:** · **tangency — first half:** · **second half:** · **1/N — second half:** · **market — second half:** · **tangency's volatility — second half:**

**(a)** Which world from Part 3 does the second half look more like?

**(b)** You set 10% volatility with the covariance matrix — the input that is
supposed to be easy to estimate. Why did you miss?

### Q8 — The memo

> **📝 Maximum ten sentences.**
>
> Your PM wants to run the tangency portfolio across the 49 industries at 10%
> volatility, on the strength of its 1.20 Sharpe ratio.
>
> - Say what 57 years of data can and cannot tell you about the 49 means.
> - Say what one mean off by two standard errors does to the weights, and to the
>   Sharpe ratio.
> - Say what World B showed, and what it implies about the 1.20.
> - Say what happened when the first-half portfolio was held through the second
>   half — its Sharpe ratio and its volatility.
> - Say what you would run instead, and why.

**Memo:**